# Break Through Tech AI: Nestlé 1A Group
## Stage 1: Exploratory Data Analysis (EDA)

As of 09/06/2025, the Nestlé 1A group has decided to use a subset of the [Amazon Reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) dataset collected in 2023 by McAuley Lab. The entire dataset consists of 571.54 M examples. We are using the raw data from the "Grocery and Gourmet Food" category, which consists of over 14 M examples, to begin the EDA stage to uncover emerging food trends from data for Nestlé and its subsidiaries.

Note: Outputs have been cleared due to rendering issues. Please run on your personal machine.

#### Step 0. Update, Install, and Import Python Libraries

In [2]:
#%pip install --upgrade pip
#%pip install -q datasets huggingface_hub pyarrow pandas
#%pip install matplotlib
#%pip install seaborn
#%pip install scikit-learn
#%pip install seaborn



In [3]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

import pandas as pd
import pyarrow as pa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

#### Step 1. Upload User Reviews and Item Metadata files from HuggingFace

In [4]:
REV_PATH = "raw/review_categories/Grocery_and_Gourmet_Food.jsonl"
META_PATH = "raw/meta_categories/meta_Grocery_and_Gourmet_Food.jsonl"

In [5]:
rev_file = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023", filename=REV_PATH, repo_type="dataset",)

In [6]:
ds_reviews = load_dataset("json", data_files=rev_file, split="train")

#### Step 2. Convert `ds_reviews` (type=datasets.arrow_dataset.Dataset) to a Pandas DataFrame for manipulation.

In [7]:
type(ds_reviews)

datasets.arrow_dataset.Dataset

In [8]:
# Note: This cell may take a while to run.
df = ds_reviews.to_pandas()

#### Step 3. Preliminary Data Analysis

In [9]:
# Display the shape of df -- that is, the number of rows and columns.
df.shape

(14318520, 10)

In [10]:
# Display the first few rows of the dataframe
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Excellent! Yummy!,Excellent!! Yummy! Great with other foods and...,[],B00CM36GAQ,B00CM36GAQ,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587854482395,0,True
1,5.0,Delicious!!! Yum!,Excellent! The best! I use it with my beef a...,[],B074J5WVYH,B0759B7KLH,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587854400380,0,True
2,5.0,"Extremely Delicious, but expensive imo",These are very tasty. They are extremely soft ...,[],B079TRNVHX,B079TRNVHX,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587853224527,1,True
3,5.0,Delicious!,My favorite!,[],B07194LN2Z,B07194LN2Z,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1581313319614,0,True
4,5.0,Great taste,Great for making brownies and crinkle cookies.,[],B005CD4196,B005CD4196,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1581313294965,7,True


In [11]:
# Display the data types of all columns
df.dtypes

rating               float64
title                 object
text                  object
images                object
asin                  object
parent_asin           object
user_id               object
timestamp              int64
helpful_vote           int64
verified_purchase       bool
dtype: object

#### Step 4. Data Cleaning: Remove Duplicates & Handle Missing Values


In [12]:
print("Shape before dropping duplicates:", df.shape)

Shape before dropping duplicates: (14318520, 10)


In [13]:
# Check for duplicate reviews based on user_id, asin, and text
dup_count = df.duplicated(subset=["user_id", "asin", "text"]).sum()
print("Total duplicate reviews:", dup_count)

Total duplicate reviews: 130960


In [14]:
# Drop duplicate reviews
df = df.drop_duplicates(subset=["user_id", "asin", "text"])
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (14187560, 10)


In [15]:
# Count missing ratings
print("Missing ratings:", df['rating'].isna().sum())

Missing ratings: 0


In [16]:
# Drop missing ratings
df = df.dropna(subset=['rating'])

In [17]:
# Remove missing or empty review text
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip() != '']

In [18]:
# Fill missing helpful_vote with 0
df['helpful_vote'] = df['helpful_vote'].fillna(0)

# Drop rows with missing verified_purchase
df = df.dropna(subset=['verified_purchase'])

In [19]:
print("Final dataset shape:", df.shape)
print(df.isna().sum())

Final dataset shape: (14170982, 10)
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64


#### Data Fields for User Reviews

| Field            | Type   | Explanation |
| :--------------- | :----- | :---------- |
| rating           | float  | Rating of the product (from 1.0 to 5.0). |
| text             | str    | Text body of the user review. |
| images           | list   | Images that users post after they have received the product.<br><br>Note: Each image has different sizes (small, medium, large), represented by the `small_image_url`, `medium_image_url`, and `large_image_url` respectively. |
| asin             | str    | ID of the product. |
| parent_asin      | str    | Parent ID of the product.<br><br>Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta. |
| user_id          | str    | ID of the reviewer. |
| timestamp        | int    | Time of the review (unix time). |
| verified_purchase| bool   | User purchase verification. |
| helpful_vote     | int    | Helpful votes of the review. |

#### Define the Label

* Do we want to have **binary classification problem** in which we will predict whether a review is a positive or negative one? We would have to create a new binary column for the label, namely `positive_review`.
* Do we want to have a **multiclass classification problem** in which we will predict the `rating` of a review from 1.0, 2.0, 3.0, 4.0, or 5.0 using the other columns? The label in this case would be `rating`.

#### Identify Features & Preliminary Observations

* Possible features include `text`, `images`, `asin`, `parent_asin`, `user_id`, `timestamp`, `verified_purchase`, `helpful_vote`
* Prior to feature engineering and feature selection, I hypothesize that we will remove `user_id` as it is merely an identification (ID) column for the user that is reviewing the product.
* The following subset of features, `asin`, `parent_asin`, and `timestamp`, may prove to be useful for a time-series analysis for reviews of a specific product over time.

#### Next Steps
* Defining the type of Machine Learning Problem.
* Defining the Label & Features
* Implementing a TF-IDF Vectorizer for the to transform data in `text` column.

#### Step 5. Determining the Label

Our goal is to predict culinary trends through time-series and sentiment analysis of this dataset. Therefore, to determine the label, we will use ordinal binning to convert the numerical `rating` column into a categorical label. In this case we will use `rating = [1.0, 2.0]` to make a new `negative` column, `rating = [3.0]` to make a new `neutral` column, and `rating = [4.0, 5.0]` to make a new `positive` column.

In [24]:
df['rating'].unique()

array([5., 4., 1., 2., 3., 0.])

In [26]:
df['sentiment'] = df['rating'].apply(
    lambda x: 'negative' if x <= 2 else ('neutral' if x == 3 else 'positive')
)